In [1]:
# 1. 必要なライブラリのインストール
!pip install ripser persim umap-learn scipy scikit-learn

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
import umap
from ripser import ripser
from persim import plot_diagrams, wasserstein
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# 2. 実験設定とディレクトリ作成
OUT_DIR = "H5_experiment_results"
os.makedirs(OUT_DIR, exist_ok=True)

# 試行回数（総生成要素数）の6分割スケール
trial_scales = [1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]

N_SAMPLES = 200 # TDAの計算負荷を考慮して200に設定
results = []

print("=== H5計算実験（コサイン距離の幾何学的誤謬の証明）を開始します ===")

# Aitchison距離のためのCLR（Centered Log-Ratio）変換関数
def clr_transform(X, epsilon=1e-10):
    # ゼロ要素を回避するための微小値加算
    X_eps = X + epsilon
    # 行ごとの幾何平均の対数を引く
    log_X = np.log(X_eps)
    return log_X - np.mean(log_X, axis=1, keepdims=True)

# 3. メインループ
for i, total_trials in enumerate(trial_scales):
    step = i + 1
    D_features = max(10, total_trials // N_SAMPLES)
    print(f"\n[Step {step}/6] 試行回数: {total_trials:,} (N={N_SAMPLES}, D={D_features})")

    # --- A. 初期値依存性を排除したベースライン生成 ---
    np.random.seed(42 + step)
    X_raw = np.random.uniform(0, 0.2, size=(N_SAMPLES, D_features))

    # --- B. 過分散・偏りの注入（ハブ化する次元の作成） ---
    X_skewed = X_raw.copy()
    hub_dims = max(1, int(D_features * 0.05))
    skew_multiplier = 10.0 + (step * 5.0)
    X_skewed[:, :hub_dims] *= skew_multiplier

    # --- C. 組成データ（確率単体）への正規化 ---
    # テキスト頻度データ（全体に占める割合）の性質を数理的に再現（総和を1にする）
    X_base_simplex = X_raw / np.sum(X_raw, axis=1, keepdims=True)
    X_skewed_simplex = X_skewed / np.sum(X_skewed, axis=1, keepdims=True)

    # --- D. 2つの距離行列の計算（コサイン距離 vs Aitchison距離） ---
    # 1. 従来のコサイン距離
    dist_cos_base = pairwise_distances(X_base_simplex + 1e-10, metric='cosine')
    dist_cos_skewed = pairwise_distances(X_skewed_simplex + 1e-10, metric='cosine')

    # 2. Aitchison距離 (CLR変換後のユークリッド距離)
    clr_base = clr_transform(X_base_simplex)
    clr_skewed = clr_transform(X_skewed_simplex)
    dist_aitchison_base = pairwise_distances(clr_base, metric='euclidean')
    dist_aitchison_skewed = pairwise_distances(clr_skewed, metric='euclidean')

    # --- E. UMAPによる次元圧縮の比較 ---
    reducer_cos = umap.UMAP(n_components=2, metric='precomputed', random_state=42)
    umap_cos = reducer_cos.fit_transform(dist_cos_skewed)

    reducer_aitch = umap.UMAP(n_components=2, metric='precomputed', random_state=42)
    umap_aitch = reducer_aitch.fit_transform(dist_aitchison_skewed)

    # --- F. TDA（パーシステントホモロジー）による位相的崩壊の定量化 ---
    ph_cos_base = ripser(dist_cos_base, distance_matrix=True, maxdim=1)['dgms']
    ph_cos_skewed = ripser(dist_cos_skewed, distance_matrix=True, maxdim=1)['dgms']

    ph_aitch_base = ripser(dist_aitchison_base, distance_matrix=True, maxdim=1)['dgms']
    ph_aitch_skewed = ripser(dist_aitchison_skewed, distance_matrix=True, maxdim=1)['dgms']

    # Wasserstein距離 (H0) を計算（ベースラインからの崩壊度合い）
    dgm_cos_base_h0 = ph_cos_base[0] if len(ph_cos_base[0]) > 0 else np.array([[0, 0]])
    dgm_cos_skew_h0 = ph_cos_skewed[0] if len(ph_cos_skewed[0]) > 0 else np.array([[0, 0]])
    w_dist_cos = wasserstein(dgm_cos_base_h0, dgm_cos_skew_h0, matching=False)

    dgm_aitch_base_h0 = ph_aitch_base[0] if len(ph_aitch_base[0]) > 0 else np.array([[0, 0]])
    dgm_aitch_skew_h0 = ph_aitch_skewed[0] if len(ph_aitch_skewed[0]) > 0 else np.array([[0, 0]])
    w_dist_aitch = wasserstein(dgm_aitch_base_h0, dgm_aitch_skew_h0, matching=False)

    # 結果の記録
    results.append({
        'Step': step,
        'Dimensions': D_features,
        'Wasserstein_H0_Cosine': w_dist_cos,
        'Wasserstein_H0_Aitchison': w_dist_aitch
    })

    # --- G. グラフ出力 ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'Step {step}: Metric Comparison on Skewed Data (D={D_features})', fontsize=16)

    # UMAP散布図
    axes[0,0].scatter(umap_cos[:,0], umap_cos[:,1], s=10, alpha=0.6, color='red')
    axes[0,0].set_title(f'UMAP (Cosine Metric)\nTopology Collapsed')
    axes[0,1].scatter(umap_aitch[:,0], umap_aitch[:,1], s=10, alpha=0.6, color='green')
    axes[0,1].set_title(f'UMAP (Aitchison Metric)\nTopology Preserved')

    # パーシステンス図
    plot_diagrams(ph_cos_skewed, ax=axes[1,0])
    axes[1,0].set_title(f'PD Skewed (Cosine Metric)\nW-Dist H0: {w_dist_cos:.3f}')

    plot_diagrams(ph_aitch_skewed, ax=axes[1,1])
    axes[1,1].set_title(f'PD Skewed (Aitchison Metric)\nW-Dist H0: {w_dist_aitch:.3f}')

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/trial_step_{step}.png")
    plt.close()

# 4. 集計とまとめグラフ
df_results = pd.DataFrame(results)
df_results.to_csv(f"{OUT_DIR}/summary_results.csv", index=False)

plt.figure(figsize=(9, 6))
plt.plot(trial_scales, df_results['Wasserstein_H0_Cosine'], marker='o', color='red', label='Cosine Distance (L2 norm vulnerability)')
plt.plot(trial_scales, df_results['Wasserstein_H0_Aitchison'], marker='s', color='green', label='Aitchison Distance (Robust to Compositional Skew)')
plt.xscale('log')
plt.xlabel('Total Trials (log scale)')
plt.ylabel('Topological Collapse (Wasserstein H0)')
plt.title('H5: Cosine Distance as the Culprit of Topological Collapse')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/summary_h5_plot.png")
plt.close()

# 5. ZIP圧縮とダウンロード
shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print(f"\n実験完了。全データを {OUT_DIR}.zip に圧縮しました。ダウンロードを開始します...")
files.download(f"{OUT_DIR}.zip")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.1/842.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.0 MB/s eta 0:00:00
  Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18187 sha256=d7e41c3780d1f3b99592fef80ae0dc525025869ffbce71881cfef04f73189e11
  Stored in directory: /root/.cache/pip/wheels/b3/32/77/3c78fadb9f917d5a5546ed0409574012401a83c544ec95999a
Successfully built hopcroftkarp
=== H5計算実験（コサイン距離の幾何学的誤謬の証明）を開始します ===

[Step 1/6] 試行回数: 1,000 (N=200, D=10)

[Step 2/6] 試行回数: 10,000 (N=200, D=50)

[Step 3/6] 試行回数: 100,000 (N=200, D=500)

[Step 4/6] 試行回数: 1,000,000 (N=200, D=5000)

[Step 5/6] 試行回数: 5,000,000 (N=200, D=25000)

[Step 6/6] 試行回数: 10,000,000 (N=200, D=50000)

実験完了。全データを H5_experiment_results.zip に圧縮しました。ダウンロードを開始します...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>